Modified version of Mohapatra et al. 2022's Single Task Learning Model. Using Whisper instead of Wav2vec. and using a CNN -> Transformer architecture instead of CNN -> FC. 

In [1]:
import pandas as pd
import torch
import librosa
import os
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm # visualization
import numpy as np 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # for training
from sklearn.utils.class_weight import compute_class_weight # class weights
from sklearn.model_selection import train_test_split # split data
from sklearn.preprocessing import LabelEncoder #transform labels
import matplotlib.pyplot as plt # visualization
import seaborn as sns # visualization
from sklearn.metrics import confusion_matrix, classification_report # visualization
import copy # copy and save model 
import math
import gc

In [2]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
device = torch.device(device)
print("Using device:", device)

Using device: cuda


In [4]:
#Whisper Model Setup
model_name = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name)
model.to(device)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 512)
    (layers): ModuleList(
      (0-5): 6 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
        (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )


In [26]:
def get_whisper_embedding_sequence(audio_path):
    if not os.path.exists(audio_path): return None
    try:
        # Load audio at 16k Hz
        speech_array, sampling_rate = librosa.load(audio_path, sr=16000)
        inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        
        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            
        # remove squeeze dimension
        embedding = encoder_outputs.last_hidden_state.squeeze(0).cpu().numpy()
        enc = embedding[:150]
        return enc 
    except Exception as e:
        print(f"Error: {e}")
        return None

In [17]:
path =     "C:/MyStuff/SJSU/Spring 2026/Ling 230/DisfluencyProject/disfluency_detection/ksh/stl/test_data/blocks/HeStutters_4_32.wav"

In [27]:
embedding = get_whisper_embedding_sequence(
    "C:/MyStuff/SJSU/Spring 2026/Ling 230/DisfluencyProject/disfluency_detection/ksh/stl/test_data/blocks/HeStutters_4_32.wav"
)

In [28]:
import librosa
y, sr = librosa.load(path, sr=16000)
print("samples:", len(y), "sr:", sr, "seconds:", len(y)/sr)

samples: 48000 sr: 16000 seconds: 3.0


In [29]:
print (embedding.shape)

(150, 512)


In [ ]:
train_path_stutter = 'train_data/blocks'
train_path_fluent  = 'train_data/fluent'
test_path_stutter  = 'test_data/blocks'
test_path_fluent   = 'test_data/fluent'